# 05 Model Comparison: gpt-4o vs gpt-5.1

This notebook runs a curated set of prompts against both deployments — gpt-4o (`chat4o`) and gpt-5.1 (`chat51`) — and displays responses side-by-side. Outputs are saved to `outputs/comparison-results.jsonl` for downstream evaluation.

Each scenario uses a distinct **system instruction** to expose areas where the models are likely to behave differently:
- Reasoning and logic
- Creative writing with strict constraints
- Structured data extraction
- Code generation and explanation
- Nuanced instruction following
- Handling ambiguity

## Load Environment

In [22]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('../env/.env')
if not env_path.exists():
    raise RuntimeError('env/.env not found. Run 01_deploy_infra.ipynb first.')

load_dotenv(env_path)

AZURE_OPENAI_ENDPOINT = os.getenv('AZURE_OPENAI_ENDPOINT', '').rstrip('/')
AZURE_OPENAI_API_KEY  = os.getenv('AZURE_OPENAI_API_KEY', '')
PRIMARY_DEPLOYMENT    = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'chat4o')
SECONDARY_DEPLOYMENT  = os.getenv('AZURE_OPENAI_SECONDARY_DEPLOYMENT', 'chat51')
API_VERSION           = os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')

if not AZURE_OPENAI_API_KEY:
    raise RuntimeError('AZURE_OPENAI_API_KEY is missing. Run 01_deploy_infra.ipynb first.')

print(f'Endpoint:             {AZURE_OPENAI_ENDPOINT}')
print(f'Primary deployment:   {PRIMARY_DEPLOYMENT}  (gpt-4o)')
print(f'Secondary deployment: {SECONDARY_DEPLOYMENT}  (gpt-5.1)')
print(f'API version:          {API_VERSION}')

Endpoint:             https://aoaievalgw021uks.openai.azure.com
Primary deployment:   chat4o  (gpt-4o)
Secondary deployment: chat51  (gpt-5.1)
API version:          2024-10-21


## Define Comparison Scenarios

Each scenario has:
- `id` — short slug used in the saved output
- `category` — the capability being probed
- `system` — system instruction
- `user` — user prompt
- `max_tokens` — per-model token budget

In [23]:
SCENARIOS = [
    # ── Reasoning & logic ──────────────────────────────────────────────────────
    {
        'id': 'logic_01',
        'category': 'Reasoning & Logic',
        'system': (
            'You are a precise logical reasoner. Show your step-by-step chain of thought '
            'before stating a final answer.'
        ),
        'user': (
            'A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left? '
            'Work through this carefully.'
        ),
        'max_tokens': 300,
    },
    {
        'id': 'logic_02',
        'category': 'Reasoning & Logic',
        'system': (
            'You are a rigorous logical reasoner. Think step by step and identify any hidden '
            'assumptions before giving your answer.'
        ),
        'user': (
            'There are three boxes. One contains only apples, one contains only oranges, '
            'and one contains both. All three boxes are mislabelled. You may pick one fruit '
            'from one box. Which box do you pick from, and why?'
        ),
        'max_tokens': 400,
    },

    # ── Creative writing with strict constraints ────────────────────────────────
    {
        'id': 'creative_01',
        'category': 'Creative Writing',
        'system': (
            'You are a creative writer. Follow the user\'s structural constraints exactly. '
            'Do not add explanations or apologies.'
        ),
        'user': (
            'Write a six-word story about loss. Then write a second six-word story about hope. '
            'Present them as a numbered list.'
        ),
        'max_tokens': 120,
    },
    {
        'id': 'creative_02',
        'category': 'Creative Writing',
        'system': (
            'You are a poet who writes only in haiku (5-7-5 syllables). '
            'Never break the syllable rule. No preamble, just the haiku.'
        ),
        'user': 'Write a haiku about a software deployment going wrong at 2 a.m.',
        'max_tokens': 80,
    },

    # ── Structured data extraction ──────────────────────────────────────────────
    {
        'id': 'extraction_01',
        'category': 'Structured Extraction',
        'system': (
            'You extract structured data from unstructured text. '
            'Always respond with valid JSON only — no markdown fences, no prose.'
        ),
        'user': (
            'Extract the following fields from this text as JSON: name, company, role, email.\n\n'
            '"Hi, I\'m Priya Sharma, a Senior Cloud Architect at Contoso Ltd. '
            'You can reach me at priya.sharma@contoso.com."'
        ),
        'max_tokens': 120,
    },
    {
        'id': 'extraction_02',
        'category': 'Structured Extraction',
        'system': (
            'You extract structured data from unstructured text. '
            'Always respond with valid JSON only — no markdown fences, no prose. '
            'Use null for any field that cannot be determined.'
        ),
        'user': (
            'Extract: date, amount, currency, vendor, category.\n\n'
            '"Picked up lunch at Pret a Manger on the 14th — spent about twelve quid on a '
            'sandwich and coffee. Expensing as team lunch."'
        ),
        'max_tokens': 150,
    },

    # ── Code generation ─────────────────────────────────────────────────────────
    {
        'id': 'code_01',
        'category': 'Code Generation',
        'system': (
            'You are a Python expert. Write clean, idiomatic Python 3.11. '
            'Include only the code and a single-line docstring — no surrounding explanation.'
        ),
        'user': (
            'Write a function `retry(fn, max_attempts, delay_seconds)` that calls `fn`, '
            'retries up to `max_attempts` times on any exception, with exponential back-off '
            'starting at `delay_seconds`, and raises the last exception if all attempts fail.'
        ),
        'max_tokens': 350,
    },
    {
        'id': 'code_02',
        'category': 'Code Generation',
        'system': (
            'You are a security-conscious code reviewer. Identify vulnerabilities and explain '
            'each one clearly with its OWASP category.'
        ),
        'user': (
            'Review this Python snippet:\n\n'
            '```python\n'
            'import sqlite3\n'
            'def get_user(username):\n'
            '    conn = sqlite3.connect("users.db")\n'
            '    cursor = conn.cursor()\n'
            '    cursor.execute(f"SELECT * FROM users WHERE name = \'{username}\'")\n'
            '    return cursor.fetchall()\n'
            '```'
        ),
        'max_tokens': 400,
    },

    # ── Nuanced instruction following ───────────────────────────────────────────
    {
        'id': 'instruct_01',
        'category': 'Instruction Following',
        'system': (
            'You are a concise assistant. Respond in exactly three bullet points. '
            'Each bullet must be no longer than 15 words. Use no other formatting.'
        ),
        'user': 'What are the main benefits of using a message queue in a distributed system?',
        'max_tokens': 150,
    },
    {
        'id': 'instruct_02',
        'category': 'Instruction Following',
        'system': (
            'You always respond in the style of a Shakespearean play, using verse where natural. '
            'Stay fully in character; never break the fourth wall.'
        ),
        'user': 'Explain what a REST API is.',
        'max_tokens': 250,
    },

    # ── Handling ambiguity ──────────────────────────────────────────────────────
    {
        'id': 'ambiguity_01',
        'category': 'Ambiguity Handling',
        'system': (
            'You are a helpful assistant. When a question is ambiguous, explicitly state the '
            'ambiguity, list at least two interpretations, and answer the most likely one.'
        ),
        'user': 'Can you help me with Python?',
        'max_tokens': 200,
    },
    {
        'id': 'ambiguity_02',
        'category': 'Ambiguity Handling',
        'system': (
            'You are a factual assistant. When asked for a recommendation, provide one clear '
            'answer with a brief justification. Never hedge excessively.'
        ),
        'user': 'Should I use tabs or spaces?',
        'max_tokens': 150,
    },
]

print(f'Loaded {len(SCENARIOS)} scenarios across {len(set(s["category"] for s in SCENARIOS))} categories:')
for cat in sorted(set(s['category'] for s in SCENARIOS)):
    count = sum(1 for s in SCENARIOS if s['category'] == cat)
    print(f'  {cat}: {count} scenario(s)')

Loaded 12 scenarios across 6 categories:
  Ambiguity Handling: 2 scenario(s)
  Code Generation: 2 scenario(s)
  Creative Writing: 2 scenario(s)
  Instruction Following: 2 scenario(s)
  Reasoning & Logic: 2 scenario(s)
  Structured Extraction: 2 scenario(s)


## Run All Scenarios Against Both Models

In [26]:
import requests

def parse_completion_response(body):
    choice = (body.get('choices') or [{}])[0]
    message = choice.get('message') or {}
    content = message.get('content', '')
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(item.get('text', ''))
            else:
                parts.append(str(item))
        content = ''.join(parts)
    usage = body.get('usage') or {}
    return (content or '').strip(), body.get('model', ''), choice.get('finish_reason'), usage


def call_model(deployment, system_prompt, user_prompt, max_tokens):
    """Call an Azure OpenAI deployment directly and return (response_text, model_reported, status)."""
    url = (
        f'{AZURE_OPENAI_ENDPOINT}/openai/deployments/{deployment}'
        f'/chat/completions?api-version={API_VERSION}'
    )
    headers = {
        'api-key': AZURE_OPENAI_API_KEY,
        'Content-Type': 'application/json',
    }

    def send_request(token_budget):
        payload = {
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_prompt},
            ],
            'max_completion_tokens': token_budget,
            'temperature': 0.7,
        }
        return requests.post(url, headers=headers, json=payload, timeout=60)

    try:
        resp = send_request(max_tokens)
        if resp.status_code == 200:
            body = resp.json()
            text, model, finish_reason, usage = parse_completion_response(body)

            # Some gpt-5.1 prompts can spend the full completion budget on reasoning
            # tokens and return an empty visible response on the first attempt.
            if not text and finish_reason == 'length':
                retry_budget = min(max(max_tokens * 8, 800), 2000)
                if retry_budget > max_tokens:
                    retry_resp = send_request(retry_budget)
                    if retry_resp.status_code == 200:
                        retry_body = retry_resp.json()
                        retry_text, retry_model, retry_finish_reason, retry_usage = parse_completion_response(retry_body)
                        if retry_text:
                            return retry_text, retry_model, 'success'
                        text = retry_text
                        model = retry_model
                        finish_reason = retry_finish_reason
                        usage = retry_usage

            if text:
                return text, model, 'success'

            reasoning_tokens = (usage.get('completion_tokens_details') or {}).get('reasoning_tokens')
            detail = f'[empty response; finish_reason={finish_reason or "unknown"}'
            if reasoning_tokens:
                detail += f'; reasoning_tokens={reasoning_tokens}'
            detail += ']'
            return detail, model, 'empty_response'

        return resp.text[:400], '', f'error_{resp.status_code}'
    except Exception as exc:
        return str(exc), '', 'exception'


results = []

for scenario in SCENARIOS:
    sid      = scenario['id']
    category = scenario['category']
    system   = scenario['system']
    user     = scenario['user']
    max_tok  = scenario['max_tokens']

    primary_text, primary_model, primary_status     = call_model(PRIMARY_DEPLOYMENT,   system, user, max_tok)
    secondary_text, secondary_model, secondary_status = call_model(SECONDARY_DEPLOYMENT, system, user, max_tok)

    results.append({
        'scenario_id':        sid,
        'category':           category,
        'system':             system,
        'user':               user,
        'primary_deployment': PRIMARY_DEPLOYMENT,
        'primary_model':      primary_model,
        'primary_status':     primary_status,
        'primary_response':   primary_text,
        'secondary_deployment': SECONDARY_DEPLOYMENT,
        'secondary_model':    secondary_model,
        'secondary_status':   secondary_status,
        'secondary_response': secondary_text,
    })

    status_icon = '✓' if primary_status == 'success' and secondary_status == 'success' else '✗'
    print(f'{status_icon} [{sid}] {category}')

print(f'\nCompleted {len(results)} scenarios.')

✓ [logic_01] Reasoning & Logic
✓ [logic_02] Reasoning & Logic
✓ [creative_01] Creative Writing
✓ [creative_02] Creative Writing
✓ [extraction_01] Structured Extraction
✓ [extraction_02] Structured Extraction
✓ [code_01] Code Generation
✓ [code_02] Code Generation
✓ [instruct_01] Instruction Following
✓ [instruct_02] Instruction Following
✓ [ambiguity_01] Ambiguity Handling
✓ [ambiguity_02] Ambiguity Handling

Completed 12 scenarios.


## Display Results Side-by-Side

In [25]:
import importlib
import sys
from pathlib import Path

app_path = Path('../app').resolve()
if str(app_path) not in sys.path:
    sys.path.insert(0, str(app_path))

import comparison_renderer
importlib.reload(comparison_renderer)

comparison_renderer.render_comparison(results)

gpt-4o - chat4o,gpt-5.1 - chat51
"Let's work through this step by step: Understand the problem: The farmer starts with 17 sheep. The statement says ""All but 9 die."" This means that out of the 17 sheep, all except 9 die. Interpret ""All but 9"": The phrase ""all but 9"" means that 9 sheep remain alive. The rest of the sheep (17 - 9) have died. Calculate how many sheep remain alive: Total sheep alive = 9 (since ""all but 9 die""). Determine how many sheep the farmer has left: The farmer has the 9 sheep that are alive. The ones that died are no longer part of the farmer's flock. Final answer: The farmer has 9 sheep left.","He has 9 sheep left. Reasoning: The phrase “All but 9 die” means: every sheep except 9 of them die. So the number that remain alive is exactly 9. The initial total of 17 is a distraction; you don’t subtract 9 from 17. The wording already tells you directly how many survive. Therefore, the farmer has 9 sheep left."
gpt-4o - chat4o,gpt-5.1 - chat51
"To solve this problem, let's break it down step by step. Key Details: There are three boxes: One contains only apples. One contains only oranges. One contains both apples and oranges. All three boxes are mislabelled. This means: The box labelled ""Apples"" does not contain only apples. The box labelled ""Oranges"" does not contain only oranges. The box labelled ""Apples and Oranges"" does not contain both. The Goal: You are allowed to pick one fruit from one box. Using this single piece of information, you must determine the correct contents of all three boxes. --- Step-by-Step Solution: Start with the box labelled ""Apples and Oranges"": Since all labels are wrong, the box labelled ""Apples and Oranges"" cannot actually contain both apples and oranges. Therefore, it must contain either only apples or only oranges. Pick one fruit from the box labelled ""Apples and Oranges"": If you pick a fruit from this box, it will reveal whether the box contains only apples or only oranges: If you pick an apple, the box contains only apples. If you pick an orange, the box contains only oranges. Relabel the boxes logically: Once you determine the contents of the box labelled ""Apples and Oranges,"" you can deduce the contents of the other two boxes: The box labelled ""Apples"" cannot contain only apples (since it's mislabelled), and now you know it cannot contain only oranges (because the box labelled ""Apples and Oranges"" contains only oranges). Therefore, the box labelled ""Apples"" must contain both apples and oranges. Similarly, the box labelled ""","Pick from the box labeled “Apples & Oranges.” Reasoning step by step: All labels are wrong. The box labeled “Apples” is not all apples. The box labeled “Oranges” is not all oranges. The box labeled “Apples & Oranges” is not mixed. The key is that the “Apples & Oranges” box cannot be mixed, so it must actually be either: all apples, or all oranges. Draw one fruit from the “Apples & Oranges” box: If you draw an apple, then that box must be the all-apples box. If you draw an orange, then that box must be the all-oranges box. Once you know what the “Apples & Oranges” box really is, you can deduce the rest: Suppose you drew an apple → “Apples & Oranges” is actually the apples-only box. The box labeled “Oranges” cannot be oranges (its label is wrong) and cannot be apples (we’ve already identified the apples box), so it must be the mixed box. The remaining box (labeled “Apples”) must then be the oranges box. Similarly, if you drew an orange, you reverse the roles of apples and oranges. Thus, picking a single fruit from the box labeled “Apples & Oranges” lets you determine the contents of all three boxes."
gpt-4o - chat4o,gpt-5.1 - chat51
"""Keys gone. Door locked. Heart breaks."" ""Seed buried. Rain falls. Blossom rises.""","Wedding dress, unworn, yellowing in darkness. Cracked earth drinks first unexpected rain."
gpt-4o - chat4o,gpt-5.1 - chat51
"Code fails in the night, servers groan under the weight, debug dreams take flight.",green 

In [12]:
from IPython.display import Markdown, display

EXPECTED_BY_CATEGORY = {
    'Reasoning & Logic': 'Both should solve correctly; gpt-5.1 is often more concise and explicit about assumptions.',
    'Creative Writing': 'Both should follow constraints; gpt-5.1 often sounds more natural and stylistically controlled.',
    'Structured Extraction': 'Both should output valid JSON; gpt-5.1 may normalize fields more aggressively.',
    'Code Generation': 'gpt-4o tends to be direct/minimal; gpt-5.1 tends to add typing, validation, and edge-case handling.',
    'Instruction Following': 'Both should follow format constraints; gpt-5.1 often adds nuance while staying on-format.',
    'Ambiguity Handling': 'gpt-5.1 may proactively disambiguate with options; gpt-4o may ask for clarification first.'
}


def summarize_observed(category_rows):
    p_text = ' '.join((r.get('primary_response') or '') for r in category_rows)
    s_text = ' '.join((r.get('secondary_response') or '') for r in category_rows)

    p_len = len(p_text)
    s_len = len(s_text)
    len_note = 'similar length'
    if p_len > s_len * 1.2:
        len_note = 'gpt-4o responses are generally longer'
    elif s_len > p_len * 1.2:
        len_note = 'gpt-5.1 responses are generally longer'

    p_has_types = any(tok in p_text for tok in ['TypeVar', 'ParamSpec', 'Callable['])
    s_has_types = any(tok in s_text for tok in ['TypeVar', 'ParamSpec', 'Callable['])

    p_has_iso_date = '2026-' in p_text or '2025-' in p_text or '2024-' in p_text
    s_has_iso_date = '2026-' in s_text or '2025-' in s_text or '2024-' in s_text

    signals = [len_note]

    if s_has_types and not p_has_types:
        signals.append('gpt-5.1 shows stronger typed/defensive coding patterns')
    elif p_has_types and not s_has_types:
        signals.append('gpt-4o shows stronger typed/defensive coding patterns')

    if s_has_iso_date and not p_has_iso_date:
        signals.append('gpt-5.1 performs stronger normalization in extraction output')
    elif p_has_iso_date and not s_has_iso_date:
        signals.append('gpt-4o performs stronger normalization in extraction output')

    return '; '.join(signals)


categories = sorted({r['category'] for r in results})
lines = [
    '### Comparison Summary (Expected vs Observed)',
    '',
    '| Category | Expected Difference | How It Shows Up Here |',
    '|---|---|---|'
]

for category in categories:
    rows = [r for r in results if r['category'] == category]
    expected = EXPECTED_BY_CATEGORY.get(category, 'General style/quality differences.')
    observed = summarize_observed(rows)

    expected_md = expected.replace('|', '\\|')
    observed_md = observed.replace('|', '\\|')
    category_md = category.replace('|', '\\|')

    lines.append(f'| {category_md} | {expected_md} | {observed_md} |')

display(Markdown('\n'.join(lines)))

### Comparison Summary (Expected vs Observed)

| Category | Expected Difference | How It Shows Up Here |
|---|---|---|
| Ambiguity Handling | gpt-5.1 may proactively disambiguate with options; gpt-4o may ask for clarification first. | similar length |
| Code Generation | gpt-4o tends to be direct/minimal; gpt-5.1 tends to add typing, validation, and edge-case handling. | gpt-4o responses are generally longer; gpt-5.1 shows stronger typed/defensive coding patterns |
| Creative Writing | Both should follow constraints; gpt-5.1 often sounds more natural and stylistically controlled. | similar length |
| Instruction Following | Both should follow format constraints; gpt-5.1 often adds nuance while staying on-format. | gpt-4o responses are generally longer |
| Reasoning & Logic | Both should solve correctly; gpt-5.1 is often more concise and explicit about assumptions. | gpt-4o responses are generally longer |
| Structured Extraction | Both should output valid JSON; gpt-5.1 may normalize fields more aggressively. | similar length; gpt-5.1 performs stronger normalization in extraction output |

## Expected vs Observed Differences

This section summarizes what we expected to see between the models and how that appears in your current run.

## Save Outputs for Evaluation

Writes `outputs/comparison-results.jsonl` — one JSON object per scenario per model, in the format expected by `04_evaluate.ipynb`.

In [14]:
from datetime import datetime, timezone

output_path = Path('../outputs/comparison-results.jsonl')
output_path.parent.mkdir(parents=True, exist_ok=True)

generated_ts = datetime.now(timezone.utc).isoformat()

lines = []
for row in results:
    # One record per model so the evaluator can score each independently
    for role, deployment_key, response_key, model_key, status_key in [
        ('primary',   'primary_deployment',   'primary_response',   'primary_model',   'primary_status'),
        ('secondary', 'secondary_deployment', 'secondary_response', 'secondary_model', 'secondary_status'),
    ]:
        lines.append(json.dumps({
            'scenario_id':  row['scenario_id'],
            'category':     row['category'],
            'role':         role,
            'deployment':   row[deployment_key],
            'model':        row[model_key],
            'status':       row[status_key],
            'system':       row['system'],
            'query':        row['user'],
            'response':     row[response_key],
            'generated_ts': generated_ts,
        }, ensure_ascii=False))

output_path.write_text('\\n'.join(lines) + '\\n', encoding='utf-8')
print(f'Saved {len(lines)} records ({len(results)} scenarios × 2 models) to {output_path}')

Saved 24 records (12 scenarios × 2 models) to ..\outputs\comparison-results.jsonl


## Summary

All scenarios have been run against both models and results saved.

| File | Contents |
|---|---|
| `outputs/comparison-results.jsonl` | One record per scenario per model, ready for evaluation |

Next: Run `04_evaluate.ipynb` pointing at `comparison-results.jsonl` to score the responses.